In [1]:
import os
import copy
import time
import urllib.request
import scipy.io as sio
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score, confusion_matrix
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.stats import ttest_rel, wilcoxon, sem, t
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import lightgbm as lgb
import psutil
import warnings
import gc
import scipy 
warnings.filterwarnings("ignore")

OUTPUT_DIR = "/kaggle/working/"
CMS_DIR = os.path.join(OUTPUT_DIR, "CMs")
CKPT_DIR = os.path.join(OUTPUT_DIR, "Checkpoints")

# ==========================================
# EXPERIMENT CONFIGURATION
# ==========================================
# This script is strictly configured to execute the remaining 50% of the experiment.
TARGET_DIRECTION = "IP_TO_LOCAL" 
directions_to_run = ['Indian Pines -> Local']

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CMS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

# ==========================================
# 1. DATA LOADING & PATCHING
# ==========================================

def read_bil_file(filepath, samples=320, bands=168, dtype=np.uint16):
    filesize = os.path.getsize(filepath)
    bytes_per_pixel = np.dtype(dtype).itemsize
    lines = filesize / (samples * bands * bytes_per_pixel)
    if not lines.is_integer(): return None
    raw_data = np.fromfile(filepath, dtype=dtype)
    img_cube = raw_data.reshape((int(lines), bands, samples))
    return np.transpose(img_cube, (0, 2, 1))

def extract_disjoint_patches(img_cube, label, patch_size=15):
    h, w, c = img_cube.shape
    patches, labels = [], []
    for i in range(0, h - patch_size + 1, patch_size):
        for j in range(0, w - patch_size + 1, patch_size):
            patches.append(img_cube[i:i+patch_size, j:j+patch_size, :])
            labels.append(label)
    return patches, labels
    
def load_local_dataset(base_dir="/kaggle/input/datasets/dev123123456/local-soil-hyperspectral-dataset"):
    images = {}
    bil_files = []
    for root, _, files in os.walk(base_dir):
        for f in files:
            if f.endswith('.bil'):
                bil_files.append(os.path.join(root, f))
                
    for filepath in bil_files:
        root = os.path.dirname(filepath)
        img = read_bil_file(filepath)
        if img is None: continue
        label = -1
        if 'black' in root.lower(): label = 0
        elif 'red' in root.lower(): label = 1
        elif 'yellow' in root.lower(): label = 2
        if label != -1:
            p, l = extract_disjoint_patches(img, label, patch_size=15)
            if len(p) > 0:
                images[filepath] = (np.array(p, dtype=np.float32), np.array(l))
    return images

def load_indian_pines_spatial(patch_size=15):
    data_url = "https://raw.githubusercontent.com/gokriznastic/HybridSN/master/data/Indian_pines_corrected.mat"
    label_url = "https://raw.githubusercontent.com/gokriznastic/HybridSN/master/data/Indian_pines_gt.mat"
    data_path = os.path.join(OUTPUT_DIR, "Indian_pines_corrected.mat")
    label_path = os.path.join(OUTPUT_DIR, "Indian_pines_gt.mat")
    if not os.path.exists(data_path): urllib.request.urlretrieve(data_url, data_path)
    if not os.path.exists(label_path): urllib.request.urlretrieve(label_url, label_path)
    X_full = sio.loadmat(data_path)['indian_pines_corrected'].astype(np.float32)
    y_full = sio.loadmat(label_path)['indian_pines_gt']
    
    h, w, c = X_full.shape
    margin = patch_size // 2
    X_padded = np.pad(X_full, ((margin, margin), (margin, margin), (0, 0)), mode='reflect')
    
    patches, labels = [], []
    for i in range(h):
        for j in range(w):
            if y_full[i, j] > 0:
                patches.append(X_padded[i:i+patch_size, j:j+patch_size, :])
                labels.append(y_full[i, j] - 1)
                
    return np.array(patches, dtype=np.float32), np.array(labels)

class SpectralDataset(Dataset):
    def __init__(self, X, y, model_type='ViT', band_mean=None, band_std=None):
        self.X = X
        self.y = y
        self.model_type = model_type
        if band_mean is None or band_std is None:
            self.band_mean = np.mean(X, axis=(0, 1, 2))
            self.band_std = np.std(X, axis=(0, 1, 2)) + 1e-8
        else:
            self.band_mean = band_mean
            self.band_std = band_std
            
    def __len__(self): return len(self.X)
    
    def __getitem__(self, idx):
        x = self.X[idx]
        x = (x - self.band_mean) / self.band_std
        x = torch.as_tensor(x, dtype=torch.float32)
        
        if self.model_type in ['ViT', 'SpectralFormer-inspired']:
            x = x[x.shape[0]//2, x.shape[1]//2, :]
        else:
            x = x.permute(2, 0, 1).unsqueeze(0)
            
        return x, torch.as_tensor(self.y[idx], dtype=torch.long)

# ==========================================
# 2. MODEL ARCHITECTURES
# ==========================================

class ViT1D(nn.Module):
    def __init__(self, bands=168, classes=3, dim=128, depth=4, heads=4):
        super().__init__()
        self.bands = bands
        self.patch_embed = nn.Linear(1, dim)
        self.pos_embed = nn.Parameter(torch.randn(1, bands, dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=dim, nhead=heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.fc = nn.Linear(dim, classes)
        
    def forward(self, x):
        x = x.unsqueeze(-1)
        x = self.patch_embed(x) + self.pos_embed
        x = self.transformer(x)
        x = x.mean(dim=1)
        return self.fc(x)

class SpectralFormerBlock(nn.Module):
    def __init__(self, dim, heads=4, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(dim, mlp_hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(mlp_hidden, dim), nn.Dropout(dropout))
    def forward(self, x):
        x_norm = self.norm1(x)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x

class CrossLayerAdaptiveFusion(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.alpha = nn.Parameter(torch.tensor(0.5))
    def forward(self, x_current, x_previous):
        alpha = torch.sigmoid(self.alpha)
        return alpha * x_current + (1.0 - alpha) * x_previous

class SpectralFormer(nn.Module):
    def __init__(self, bands=168, classes=3, group_size=4, dim=64, depth=4, heads=4, dropout=0.1):
        super().__init__()
        self.bands = bands
        self.group_size = group_size
        self.num_tokens = bands // group_size
        self.effective_bands = self.num_tokens * group_size

        self.group_embed = nn.Linear(group_size, dim)
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_tokens, dim) * 0.02)
        self.blocks = nn.ModuleList([SpectralFormerBlock(dim, heads=heads, mlp_ratio=4.0, dropout=dropout) for _ in range(depth)])
        self.caf_modules = nn.ModuleList([CrossLayerAdaptiveFusion(dim) for _ in range(depth - 1)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, classes)

    def forward(self, x):
        x = x[:, :self.effective_bands]
        x = x.view(x.shape[0], self.num_tokens, self.group_size)
        x = self.group_embed(x) + self.pos_embed
        prev_x = None
        for i, block in enumerate(self.blocks):
            out = block(x)
            if i > 0: x = self.caf_modules[i-1](out, prev_x)
            else: x = out
            prev_x = x
        x = self.norm(x).mean(dim=1)
        return self.head(x)

class HybridSN(nn.Module):
    def __init__(self, bands=200, classes=16, spatial_size=15):
        super(HybridSN, self).__init__()
        self.conv1 = nn.Conv3d(in_channels=1, out_channels=8, kernel_size=(7, 3, 3))
        self.conv2 = nn.Conv3d(in_channels=8, out_channels=16, kernel_size=(5, 3, 3))
        self.conv3 = nn.Conv3d(in_channels=16, out_channels=32, kernel_size=(3, 3, 3))
        
        # Dynamic shape calculation
        dummy = torch.zeros(1, 1, bands, spatial_size, spatial_size)
        with torch.no_grad():
            x = self.conv1(dummy)
            x = self.conv2(x)
            x = self.conv3(x)
            in_2d = x.size(1) * x.size(2)
            out_H, out_W = x.size(3), x.size(4)
        
        self.conv4 = nn.Conv2d(in_channels=in_2d, out_channels=64, kernel_size=(3, 3))
        
        with torch.no_grad():
            x = x.view(x.size(0), in_2d, out_H, out_W)
            x = self.conv4(x)
            self.flat_features = x.view(x.size(0), -1).size(1)
            
        self.fc1 = nn.Linear(self.flat_features, 256)
        self.dropout1 = nn.Dropout(0.4)
        self.fc2 = nn.Linear(256, 128)
        self.dropout2 = nn.Dropout(0.4)
        self.fc3 = nn.Linear(128, classes)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = x.view(x.size(0), x.size(1) * x.size(2), x.size(3), x.size(4))
        x = F.relu(self.conv4(x))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        return self.fc3(x)

# ==========================================
# 3. TRAINING LOGIC & STATS
# ==========================================

def get_confidence_interval(data, confidence=0.95):
    n = len(data)
    m = np.mean(data)
    std_err = sem(data)
    if std_err == 0: return m, m, m
    h = std_err * t.ppf((1 + confidence) / 2, n - 1)
    return m, m - h, m + h

def get_interpolated_crossover(lgb_accs, deep_accs, splits):
    # Linear interpolation for exact crossover percentage
    for i in range(1, len(splits)):
        diff_prev = deep_accs[i-1] - lgb_accs[i-1]
        diff_curr = deep_accs[i] - lgb_accs[i]
        
        if diff_prev <= 0 and diff_curr > 0:
            slope = (diff_curr - diff_prev) / (splits[i] - splits[i-1])
            x_cross = splits[i-1] - diff_prev / slope
            # Ensure monotonicity check: if it crosses back down later, it's not stable
            for j in range(i+1, len(splits)):
                if deep_accs[j] < lgb_accs[j]:
                    return None
            return x_cross
    return None

def train_pytorch_model(model, train_loader, test_loader, epochs=50, freeze_encoder=False, unfreeze_epoch=5):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    
    if freeze_encoder:
        for name, param in model.named_parameters():
            if 'fc' not in name and 'head' not in name:
                param.requires_grad = False
    
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    best_loss = float('inf')
    best_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())
    patience = 7
    patience_counter = 0
    
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    
    start_train = time.time()
    for epoch in range(epochs):
        if freeze_encoder and epoch == unfreeze_epoch:
            for param in model.parameters(): param.requires_grad = True
            optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs - unfreeze_epoch)
            
        model.train()
        train_loss = 0
        train_preds, train_trues = [], []
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            out = model(X_batch.to(device))
            loss = criterion(out, y_batch.to(device))
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            train_preds.extend(torch.argmax(out, dim=1).cpu().numpy())
            train_trues.extend(y_batch.numpy())
            
        train_acc = accuracy_score(train_trues, train_preds)
            
        model.eval()
        val_loss = 0
        preds, trues = [], []
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                out = model(X_batch.to(device))
                loss = criterion(out, y_batch.to(device))
                val_loss += loss.item()
                preds.extend(torch.argmax(out, dim=1).cpu().numpy())
                trues.extend(y_batch.numpy())
                
        val_acc = accuracy_score(trues, preds)
        history['train_loss'].append(train_loss / max(len(train_loader), 1))
        history['val_loss'].append(val_loss / max(len(test_loader), 1))
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        scheduler.step()
        
        if val_loss < best_loss:
            best_loss = val_loss
            best_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            patience_counter = 0
        elif val_loss == best_loss and val_acc > best_acc:
            best_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            
        if patience_counter >= patience:
            break
            
    train_time = time.time() - start_train
    
    model.load_state_dict(best_model_wts)
    model.eval()
    preds, trues = [], []
    start_inf = time.time()
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            out = model(X_batch.to(device))
            preds.extend(torch.argmax(out, dim=1).cpu().numpy())
            trues.extend(y_batch.numpy())
    inf_time = time.time() - start_inf
            
    cm = confusion_matrix(trues, preds)
    class_acc = cm.diagonal() / np.maximum(cm.sum(axis=1), 1)
    
    # Resource Profiling
    vram_mb = torch.cuda.max_memory_allocated() / (1024**2) if torch.cuda.is_available() else 0
    process = psutil.Process(os.getpid())
    ram_mb = process.memory_info().rss / (1024**2)
    
    metrics = {
        'OA': accuracy_score(trues, preds),
        'AA': np.mean(class_acc),
        'Per_Class': class_acc.tolist(),
        'Kappa': cohen_kappa_score(trues, preds),
        'MacroF1': f1_score(trues, preds, average='macro'),
        'CM': cm,
        'TrainTime': train_time,
        'InfTime': inf_time,
        'VRAM_MB': vram_mb,
        'RAM_MB': ram_mb
    }
    return metrics, history, model

def transfer_weights(model_type, pretrained_model, ft_model, src_bands, tgt_bands):
    pt_dict = pretrained_model.state_dict()
    ft_dict = ft_model.state_dict()
    
    if model_type == 'ViT':
        pos = pt_dict['pos_embed'].permute(0, 2, 1)
        pos_interp = F.interpolate(pos, size=tgt_bands, mode='linear', align_corners=False)
        ft_dict['pos_embed'] = pos_interp.permute(0, 2, 1)
        pt_dict = {k: v for k, v in pt_dict.items() if k in ft_dict and 'pos_embed' not in k and 'fc' not in k}
        ft_dict.update(pt_dict)
        
    elif model_type == 'SpectralFormer-inspired':
        src_tokens, tgt_tokens = src_bands // 4, tgt_bands // 4
        pos = pt_dict['pos_embed'].permute(0, 2, 1)
        pos_interp = F.interpolate(pos, size=tgt_tokens, mode='linear', align_corners=False)
        ft_dict['pos_embed'] = pos_interp.permute(0, 2, 1)
        pt_dict = {k: v for k, v in pt_dict.items() if k in ft_dict and 'pos_embed' not in k and 'head' not in k}
        ft_dict.update(pt_dict)
        
    elif model_type == 'HybridSN':
        pt_dict = {k: v for k, v in pt_dict.items() if 'conv1' in k or 'conv2' in k or 'conv3' in k}
        if pretrained_model.fc1.weight.shape == ft_model.fc1.weight.shape:
            pt_dict.update({k: v for k, v in pretrained_model.state_dict().items() if 'fc1' in k})
        ft_dict.update(pt_dict)
        
    ft_model.load_state_dict(ft_dict)
    return ft_model

def run_all_experiments():
    base_dir = "/kaggle/input/datasets/dev123123456/local-soil-hyperspectral-dataset"
    OUTPUT_DIR = "/kaggle/working/"
    print("Loading Indian Pines Dataset...")
    X_ip, y_ip = load_indian_pines_spatial(patch_size=15)
    
    print("Loading Local Soil Dataset...")
    local_images = load_local_dataset(base_dir)
    if len(local_images) == 0:
        X_local = np.random.rand(500, 15, 15, 168).astype(np.float32)
        y_local = np.random.randint(0, 3, 500)
    else:
        # Mathematical reproducible Image-Level Split using Seed 42
        files = list(local_images.keys())
        rng = np.random.default_rng(42)
        rng.shuffle(files)
        train_files = files[:int(0.8 * len(files))]
        test_files = files[int(0.8 * len(files)):]
        
        X_train_loc, y_train_loc, X_test_loc, y_test_loc = [], [], [], []
        for f in train_files:
            X_train_loc.extend(local_images[f][0]); y_train_loc.extend(local_images[f][1])
        for f in test_files:
            X_test_loc.extend(local_images[f][0]); y_test_loc.extend(local_images[f][1])
            
        X_train_loc = np.array(X_train_loc, dtype=np.float32)
        y_train_loc = np.array(y_train_loc)
        X_test_loc = np.array(X_test_loc, dtype=np.float32)
        y_test_loc = np.array(y_test_loc)
        
        # AGGRESSIVELY FREE MEMORY TO PREVENT OOM
        del local_images
        gc.collect()

    splits = [0.01, 0.05, 0.10, 0.20]
    num_seeds = 10
    models_to_run = ['HybridSN', 'SpectralFormer-inspired', 'ViT']
    
    pretrained_ip_models = {}
    
    # ---------------------------------------------------------
    # 3. PRETRAINING ON INDIAN PINES (Source Domain)
    # ---------------------------------------------------------
    pretrained_ip_models = {}
    print("\n[PRETRAINING ON INDIAN PINES (80% Data for monitoring Reverse Transfer)]")
    
    # We create a random split just for validation tracking during pretraining
    X_train_ip_pt, X_test_ip_pt, y_train_ip_pt, y_test_ip_pt = train_test_split(X_ip, y_ip, test_size=0.2, random_state=42, stratify=y_ip)
    for m in models_to_run:
        train_ds = SpectralDataset(X_train_ip_pt, y_train_ip_pt, m)
        train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
        test_loader = DataLoader(SpectralDataset(X_test_ip_pt, y_test_ip_pt, m, band_mean=train_ds.band_mean, band_std=train_ds.band_std), batch_size=64, shuffle=False)
        model = HybridSN(200, 16) if m == 'HybridSN' else (SpectralFormer(200, 16) if m == 'SpectralFormer-inspired' else ViT1D(200, 16))
        metrics, _, trained = train_pytorch_model(model, train_loader, test_loader, epochs=30)
        pretrained_ip_models[m] = trained
        torch.save(trained.state_dict(), os.path.join(CKPT_DIR, f"Pretrained_{m}_IP.pt"))
        print(f"  Pretrained {m} OA: {metrics['OA']:.4f}")

    # ---------------------------------------------------------
    # DOWNSTREAM TRANSFER EXPERIMENTS
    # ---------------------------------------------------------
    results = []
    histories_dict = []
    csv_path = os.path.join(OUTPUT_DIR, f"EXP4_FULL_RESULTS_{TARGET_DIRECTION}.csv")
    cols_to_drop = ['Scratch_Loss', 'FT_Loss', 'Scratch_Val_Loss', 'FT_Val_Loss', 'Scratch_Val_Acc', 'FT_Val_Acc', 'Scratch_Train_Acc', 'FT_Train_Acc']
    
    def run_lgb(X_tr, y_tr, X_te, y_te, seed):
        X_tr_flat = X_tr[:, X_tr.shape[1]//2, X_tr.shape[2]//2, :]
        X_te_flat = X_te[:, X_te.shape[1]//2, X_te.shape[2]//2, :]
            
        clf = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.1, num_leaves=31, feature_fraction=1.0, bagging_fraction=1.0, bagging_freq=0, verbose=-1, random_state=seed)
        clf.fit(X_tr_flat, y_tr)
        preds = clf.predict(X_te_flat)
        return accuracy_score(y_te, preds)

    for direction in directions_to_run:
        print(f"\n{'='*50}\nDIRECTION: {direction}\n{'='*50}")
        is_ip_target = (direction == 'Local -> Indian Pines')
        
        X_tgt, y_tgt = (X_ip, y_ip) if is_ip_target else (X_train_loc, y_train_loc) # Dummy fallback
        tgt_bands, tgt_classes = (200, 16) if is_ip_target else (168, 3)
        src_bands = 168 if is_ip_target else 200
        pretrained_dict = pretrained_ip_models
        
        for split in splits:
            print(f"\n--- Split {split*100}% ---")
            for seed in range(42, 42 + num_seeds):
                
                if is_ip_target:
                    # Community Standard Protocol
                    try:
                        X_tr, X_te, y_tr, y_te = train_test_split(X_tgt, y_tgt, train_size=split, random_state=seed, stratify=y_tgt)
                    except ValueError:
                        X_tr, X_te, y_tr, y_te = train_test_split(X_tgt, y_tgt, train_size=split, random_state=seed)
                else:
                    # Target is Local. We keep test_files strictly isolated for evaluation.
                    # We sample the 'split' percentage exclusively from X_train_loc
                    try:
                        X_tr, _, y_tr, _ = train_test_split(X_train_loc, y_train_loc, train_size=split, random_state=seed, stratify=y_train_loc)
                    except ValueError:
                        X_tr, _, y_tr, _ = train_test_split(X_train_loc, y_train_loc, train_size=split, random_state=seed)
                    
                    X_te, y_te = X_test_loc, y_test_loc
                    
                lgb_oa = run_lgb(X_tr, y_tr, X_te, y_te, seed)
                
                for m in models_to_run:
                    train_ds = SpectralDataset(X_tr, y_tr, m)
                    tr_ldr = DataLoader(train_ds, batch_size=32, shuffle=True)
                    
                    test_ds = SpectralDataset(X_te, y_te, m, band_mean=train_ds.band_mean, band_std=train_ds.band_std)
                    te_ldr = DataLoader(test_ds, batch_size=32, shuffle=False)
                    
                    scratch = HybridSN(tgt_bands, tgt_classes) if m == 'HybridSN' else (SpectralFormer(tgt_bands, tgt_classes) if m == 'SpectralFormer-inspired' else ViT1D(tgt_bands, tgt_classes))
                    sc_met, sc_hist, sc_trained = train_pytorch_model(scratch, tr_ldr, te_ldr, epochs=30)
                    np.save(os.path.join(CMS_DIR, f"CM_Scratch_{m}_{direction[:3]}_{split}_{seed}.npy"), sc_met['CM'])
                    if seed == 42: torch.save(sc_trained.state_dict(), os.path.join(CKPT_DIR, f"Scratch_{m}_{direction[:3]}_{split}_seed42.pt"))
                    
                    ft = HybridSN(tgt_bands, tgt_classes) if m == 'HybridSN' else (SpectralFormer(tgt_bands, tgt_classes) if m == 'SpectralFormer-inspired' else ViT1D(tgt_bands, tgt_classes))
                    ft = transfer_weights(m, pretrained_dict[m], ft, src_bands, tgt_bands)
                    ft_met, ft_hist, ft_trained = train_pytorch_model(ft, tr_ldr, te_ldr, epochs=30, freeze_encoder=True, unfreeze_epoch=5)
                    np.save(os.path.join(CMS_DIR, f"CM_Transfer_{m}_{direction[:3]}_{split}_{seed}.npy"), ft_met['CM'])
                    if seed == 42: torch.save(ft_trained.state_dict(), os.path.join(CKPT_DIR, f"Transfer_{m}_{direction[:3]}_{split}_seed42.pt"))
                    
                    res = {
                        'Direction': direction, 'Model': m, 'Split': split, 'Seed': seed, 'LGBM_OA': lgb_oa,
                        'Scratch_OA': sc_met['OA'], 'Scratch_AA': sc_met['AA'], 'Scratch_F1': sc_met['MacroF1'],
                        'FT_OA': ft_met['OA'], 'FT_AA': ft_met['AA'], 'FT_F1': ft_met['MacroF1'],
                        'TransferGain': ft_met['OA'] - sc_met['OA'],
                        'Scratch_TrainTime': sc_met['TrainTime'], 'Scratch_InfTime': sc_met['InfTime'],
                        'FT_TrainTime': ft_met['TrainTime'], 'FT_InfTime': ft_met['InfTime'],
                        'VRAM_MB': sc_met['VRAM_MB'], 'RAM_MB': sc_met['RAM_MB'],
                        'Scratch_Loss': sc_hist['train_loss'], 'FT_Loss': ft_hist['train_loss'],
                        'Scratch_Val_Loss': sc_hist['val_loss'], 'FT_Val_Loss': ft_hist['val_loss'],
                        'Scratch_Val_Acc': sc_hist['val_acc'], 'FT_Val_Acc': ft_hist['val_acc'],
                        'Scratch_Train_Acc': sc_hist['train_acc'], 'FT_Train_Acc': ft_hist['train_acc']
                    }
                    
                    histories_dict.append({
                        'Direction': direction, 'Model': m, 'Split': split, 'Seed': seed,
                        'Scratch_Loss': sc_hist['train_loss'], 'FT_Loss': ft_hist['train_loss'],
                        'Scratch_Val_Loss': sc_hist['val_loss'], 'FT_Val_Loss': ft_hist['val_loss'],
                        'Scratch_Val_Acc': sc_hist['val_acc'], 'FT_Val_Acc': ft_hist['val_acc'],
                        'Scratch_Train_Acc': sc_hist['train_acc'], 'FT_Train_Acc': ft_hist['train_acc']
                    })
                    # Add Per Class AA
                    for cls_i in range(tgt_classes):
                        res[f'Scratch_Class_{cls_i}_AA'] = sc_met['Per_Class'][cls_i] if cls_i < len(sc_met['Per_Class']) else 0
                        res[f'FT_Class_{cls_i}_AA'] = ft_met['Per_Class'][cls_i] if cls_i < len(ft_met['Per_Class']) else 0
                        
                    results.append(res)
                    
                    # INCREMENTAL SAVING (Prevents 12-hour timeout data loss)
                    pd.DataFrame([res]).drop(columns=cols_to_drop).to_csv(
                        csv_path, mode='a', header=not os.path.exists(csv_path), index=False
                    )
                    import json
                    with open(os.path.join(OUTPUT_DIR, f"training_histories_{TARGET_DIRECTION}.json"), "w") as f:
                        json.dump(histories_dict, f)
                        
                    del scratch, ft, sc_trained, ft_trained
                    gc.collect()
                    if torch.cuda.is_available(): torch.cuda.empty_cache()

    df = pd.DataFrame(results)
    import sys
    import sklearn
    manifest = {
        'seed_list': list(range(42, 42 + num_seeds)),
        'split_percentages': splits,
        'model_versions': models_to_run,
        'python_version': sys.version,
        'pytorch_version': torch.__version__,
        'cuda_version': torch.version.cuda,
        'numpy_version': np.__version__,
        'scipy_version': scipy.__version__,
        'sklearn_version': sklearn.__version__,
        'lightgbm_version': lgb.__version__
    }
    with open(os.path.join(OUTPUT_DIR, "manifest.json"), "w") as f:
        json.dump(manifest, f)
    
    # ---------------------------------------------------------
    # STATISTICS & CROSSOVER AUTOMATION
    # ---------------------------------------------------------
    stats_log = []
    p_vals_to_correct = []
    tests_metadata = []
    
    print("\n[STATISTICS & CROSSOVER POINTS]")
    for direction in directions_to_run:
        print(f"\n--- Direction: {direction} ---")
        df_dir = df[df['Direction'] == direction]
        
        mean_lgb = df_dir.groupby('Split')['LGBM_OA'].mean().values
        for m in models_to_run:
            df_m = df_dir[df_dir['Model'] == m]
            mean_sc = df_m.groupby('Split')['Scratch_OA'].mean().values
            mean_ft = df_m.groupby('Split')['FT_OA'].mean().values
            
            cross_sc = get_interpolated_crossover(mean_lgb, mean_sc, splits)
            cross_ft = get_interpolated_crossover(mean_lgb, mean_ft, splits)
            print(f"  {m} | Scratch Crossover: {cross_sc*100 if cross_sc else 'N/A'}% | Transfer Crossover: {cross_ft*100 if cross_ft else 'N/A'}%")
            
            for split in splits:
                sub = df_m[df_m['Split'] == split]
                if len(sub) < 2: continue
                sc_arr, ft_arr = sub['Scratch_OA'].values, sub['FT_OA'].values
                mean_diff = np.mean(ft_arr - sc_arr)
                _, ci_low, ci_high = get_confidence_interval(ft_arr - sc_arr)
                
                _, p_val_t = ttest_rel(sc_arr, ft_arr)
                try:
                    _, p_val_w = wilcoxon(sc_arr, ft_arr)
                except:
                    p_val_w = 1.0
                
                std_diff = np.std(ft_arr - sc_arr, ddof=1)
                d = mean_diff / std_diff if std_diff > 0 else 0
                
                p_vals_to_correct.extend([p_val_t, p_val_w])
                tests_metadata.append({'Direction': direction, 'Model': m, 'Split': split, 'Mean_Diff': mean_diff, 'CI_Low': ci_low, 'CI_High': ci_high, 'Cohen_dz': d, 'P_Val_T': p_val_t, 'P_Val_W': p_val_w})

    # FDR Correction (Benjamini-Hochberg)
    if len(p_vals_to_correct) > 0:
        _, p_vals_fdr, _, _ = multipletests(p_vals_to_correct, alpha=0.05, method='fdr_bh')
        # We appended 2 p-values per test metadata (T and W)
        idx = 0
        for meta in tests_metadata:
            meta['P_Val_T_FDR'] = p_vals_fdr[idx]
            meta['P_Val_W_FDR'] = p_vals_fdr[idx+1]
            idx += 2
            stats_log.append(meta)
            
    pd.DataFrame(stats_log).to_csv(os.path.join(OUTPUT_DIR, "EXP4_FDR_STATS.csv"), index=False)
    
    # ---------------------------------------------------------
    # PLOTTING LOSS & ACCURACY CURVES (Across Splits)
    # ---------------------------------------------------------
    for direction in directions_to_run:
        dir_slug = direction.replace(' -> ', '_to_').replace(' ', '')
        for split in splits:
            fig_loss, ax_loss = plt.subplots(1, len(models_to_run), figsize=(18, 5))
            fig_acc, ax_acc = plt.subplots(1, len(models_to_run), figsize=(18, 5))
            
            for idx, m in enumerate(models_to_run):
                h_plot = [h for h in histories_dict if h['Direction'] == direction and h['Model'] == m and h['Split'] == split]
                if len(h_plot) == 0: continue
                
                # Means over seeds with padding for early stopping
                def pad_histories(key):
                    h_list = [h[key] for h in h_plot]
                    max_len = max(len(h) for h in h_list)
                    padded = [list(h) + [h[-1]] * (max_len - len(h)) for h in h_list]
                    return np.mean(padded, axis=0)
                
                sc_tl = pad_histories('Scratch_Loss')
                sc_vl = pad_histories('Scratch_Val_Loss')
                ft_tl = pad_histories('FT_Loss')
                ft_vl = pad_histories('FT_Val_Loss')
                
                ax_loss[idx].plot(sc_tl, label='Scratch Train Loss', linestyle='--')
                ax_loss[idx].plot(sc_vl, label='Scratch Val Loss', linestyle=':')
                ax_loss[idx].plot(ft_tl, label='Transfer Train Loss', linewidth=2)
                ax_loss[idx].plot(ft_vl, label='Transfer Val Loss', linewidth=2)
                ax_loss[idx].set_title(f"{m} (Loss)")
                ax_loss[idx].legend()
                
                sc_ta = pad_histories('Scratch_Train_Acc')
                sc_va = pad_histories('Scratch_Val_Acc')
                ft_ta = pad_histories('FT_Train_Acc')
                ft_va = pad_histories('FT_Val_Acc')
                
                ax_acc[idx].plot(sc_ta, label='Scratch Train Acc', linestyle='--')
                ax_acc[idx].plot(sc_va, label='Scratch Val Acc', linestyle=':')
                ax_acc[idx].plot(ft_ta, label='Transfer Train Acc', linewidth=2)
                ax_acc[idx].plot(ft_va, label='Transfer Val Acc', linewidth=2)
                ax_acc[idx].set_title(f"{m} (Accuracy)")
                ax_acc[idx].legend()
                
            fig_loss.suptitle(f"{direction} - Loss Curves ({split*100}%)", fontsize=16)
            fig_loss.tight_layout()
            fig_loss.savefig(os.path.join(OUTPUT_DIR, f"loss_curves_{dir_slug}_{split}.png"))
            plt.close(fig_loss)
            
            fig_acc.suptitle(f"{direction} - Accuracy Curves ({split*100}%)", fontsize=16)
            fig_acc.tight_layout()
            fig_acc.savefig(os.path.join(OUTPUT_DIR, f"accuracy_curves_{dir_slug}_{split}.png"))
            plt.close(fig_acc)

    print("\n✅ EXPERIMENT 4 COMPLETE.")

if __name__ == "__main__":
    run_all_experiments()


Loading Indian Pines Dataset...
Loading Local Soil Dataset...

[PRETRAINING ON INDIAN PINES (80% Data for monitoring Reverse Transfer)]
  Pretrained HybridSN OA: 1.0000
  Pretrained SpectralFormer-inspired OA: 0.8927
  Pretrained ViT OA: 0.8980

DIRECTION: Indian Pines -> Local

--- Split 1.0% ---

--- Split 5.0% ---

--- Split 10.0% ---

--- Split 20.0% ---

[STATISTICS & CROSSOVER POINTS]

--- Direction: Indian Pines -> Local ---
  HybridSN | Scratch Crossover: 4.853953682453581% | Transfer Crossover: 4.254425174325051%
  SpectralFormer-inspired | Scratch Crossover: N/A% | Transfer Crossover: N/A%
  ViT | Scratch Crossover: N/A% | Transfer Crossover: N/A%

✅ EXPERIMENT 4 COMPLETE.
